In [1]:
import os
import random
import time
import pickle
import pandas as pd
import geopandas as gpd
import shapely.geometry
import rasterio

from tqdm import tqdm
from streetview import StreetViewDownloader, ImageService
from IPython.display import Image, display
from typing import Dict, Tuple, List, Union, Any

from streetview import *
from socioeconomics import *
from building import *
from vs30 import *
from station import *
from prompt import *
from config import *

### 🌍 FILES INPUT

This notebook processes earthquake-related data and requires the following files based on `config.py`. The notebook will generate geospatial image samples and analysis-ready data for earthquake damage assessment. 

#### Earthquake Data

- **Community-Reported Intensity**  
  `2019_ridgecrest_DYFI.csv`  
  *"Did You Feel It?"* (DYFI) dataset containing self-reported earthquake intensity levels.

- **Seismic Station Measurements**  
  `2019-ridgecrest/stationlist.json`  
  JSON file with seismic station metadata and ground motion values.


#### Geographic Files

- **ZIP Code Boundaries**  
  `nhgis_shape/US_zcta_2019.shp`

- **Census Block Group Boundaries**  
  `nhgis_shape/US_blck_grp_2019_84.shp`

- **Socioeconomic Attributes**  
  `nhgis_shape/cbg_information.csv`  
  Contains demographic and economic indicators at the block group level.


In [2]:
df = pd.read_csv(DYFI_DATA)
df = df[df['Country'] == 'United States of America']
df = df.sort_values(by='Responses', ascending=False)
df["Zip Code"] = df["Zip Code"].apply(lambda x: str(int(x)).zfill(5))

# df1 - Generate Samples
df1 = df.iloc[:NUM_SAMPLES]

# df2 - Generate Dictionary for RAG
df2 = df.iloc[NUM_SAMPLES:][df['Responses']>=10]
df2 = df2.reset_index()
df2 = df2.drop('index', axis=1) 
print(len(df1), len(df2))
df1.head(5)

100 553


/var/folders/l0/f3brd6kd23d1j64rj2yz5r100000gp/T/ipykernel_88225/4165242244.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df2 = df.iloc[NUM_SAMPLES:][df['Responses']>=10]


,City,State/Region,Country,Zip Code,MMI,Responses,Distance,Latitude,Longitude
1027,Ridgecrest,CA,United States of America,93555,VII,314,3 km,35.691731,-117.449167
220,Las Vegas,NV,United States of America,89101,IV,204,213 km,36.172608,-115.122034
630,San Diego,CA,United States of America,92101,III,179,316 km,32.723359,-117.169286
286,Los Angeles,CA,United States of America,90012,IV,109,185 km,34.065853,-118.238610
1082,Fresno,CA,United States of America,93721,III,108,205 km,36.732871,-119.783726


In [4]:
# INPUT ALL NECESSARY FILES

eq_data = EARTHQUAKE_PARAMETERS

zcta = gpd.read_file(ZCTA_SHAPEFILE)
zcta = zcta.to_crs(epsg=4326)

socioeconomic_df = pd.read_csv(SOCIOECONOMIC_DATA)
cbg_gdf = gpd.read_file(CBG_SHAPEFILE)
cbg_gdf = cbg_gdf.to_crs(epsg=4326)

stations_df = load_station_data(STATION_DATA)
stations_df = stations_df.sort_values(by='Nresp', ascending=False)

src = rasterio.open(VS30_FILE)

### 🌍 Spatial Image Download

The following module generates random sampling points within geographic boundaries based on zipcode and downloads corresponding Google Street View images.


#### ⚙️ Functions

##### `generate_points_with_images`

Generates random points within a polygon that have valid Street View imagery.

- Handles coordinate selection, metadata recording, and image storage
- Supports resume capability and real-time progress saving


##### `download_location_images`

Downloads Google Street View images for specific coordinates.

- Manages file naming conventions
- Includes robust error handling for failed downloads


##### `generate_samples_with_images`

Processes multiple ZIP codes to collect geographic sample points with valid Street View images.

- Skips ZIP codes that already have sufficient samples
- Validates and cleans geographic input data


In [4]:
def generate_points_with_images(
    polygon, 
    downloader, 
    zip_code, 
    city, 
    state, 
    country, 
    points_needed, 
    output_csv_path,
    max_attempts=MAX_ATTEMPTS,
    image_dir=OUTPUT_IMAGES_DIR
):
    """
    Generate random points within a polygon that have valid Street View images.
    Save results to CSV in real-time as each valid point is found.
    """
    
    # Create directory for images
    os.makedirs(image_dir, exist_ok=True)

    # Check if output CSV already exists and load existing data
    existing_points = []
    existing_count = 0
    
    if os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path)
        # Filter for points from current zip code
        zip_points = existing_df[existing_df['Zip Code'] == zip_code]
        existing_count = len(zip_points)
        existing_points = zip_points.to_dict('records')
        
        # If we already have enough points, return early
        if existing_count >= points_needed:
            print(f"Already have {existing_count} points for ZIP {zip_code}. Skipping.")
            return existing_points
    
    # Calculate how many more points we need
    remaining_points = points_needed - existing_count
    
    # Create progress bar for remaining points
    pbar = tqdm(total=remaining_points, desc=f"Finding points with images in {zip_code}")
    
    valid_points = existing_points.copy()  # Start with existing points
    minx, miny, maxx, maxy = polygon.bounds
    attempts = 0
    
    while len(valid_points) < points_needed and attempts < max_attempts:
        # Generate a random point within the bounding box
        pt = shapely.geometry.Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        
        # Check if point is within the polygon
        if polygon.contains(pt):
            lat, lng = pt.y, pt.x 
            temp_id = f"{zip_code}_temp_{attempts}"
            success, result = download_location_images(
                downloader,
                lat,
                lng,
                location_id=temp_id,
                services=[ImageService.GOOGLE],
                size='640x640'
            )
            
            if success:
                # Calculate the sequential ID based on the current number of valid points
                current_count = len(valid_points) + 1
                sequential_id = f"{zip_code}_{current_count}"

                if os.path.exists(result):
                    new_path = f"{image_dir}/{sequential_id}_google_streetview_{lat}_{lng}.jpg"
                    
                    try:
                        with open(result, 'rb') as src_file:
                            content = src_file.read()
                            with open(new_path, 'wb') as dest_file:
                                dest_file.write(content)
                        if result != new_path:  # Don't try to remove if it's the same file
                            os.remove(result)
                        result = new_path
                    except Exception as e:
                        print(f"Error renaming file: {str(e)}")
                
                # Add the point with the sequential ID
                point_data = {
                    "location_id": sequential_id,
                    "City": city,
                    "State/Region": state,
                    "Country": country,
                    "Zip Code": zip_code,
                    "Latitude": lat,
                    "Longitude": lng,
                    "image_path": result
                }
                valid_points.append(point_data)
                
                # Save only the new point to the CSV
                new_df = pd.DataFrame([point_data])
                if os.path.exists(output_csv_path):
                    new_df.to_csv(output_csv_path, mode='a', header=False, index=False)
                else:
                    new_df.to_csv(output_csv_path, mode='w', header=True, index=False)
                
                pbar.update(1)
                
                # If we've collected enough points, break the loop
                if len(valid_points) >= points_needed:
                    break
                
        attempts += 1
        
        # If too many attempts, break and report
        if attempts >= max_attempts and len(valid_points) < points_needed:
            print(f"\nWarning: Could only find {len(valid_points)} valid points with images " 
                  f"for ZIP {zip_code} after {max_attempts} attempts.")
            break
    
    pbar.close()
    return valid_points


def download_location_images(
    downloader,
    latitude: float,
    longitude: float,
    location_id: str = None,
    services = None,
    size: str = '640x640',
    image_dir = OUTPUT_IMAGES_DIR
):
    """
    Download images for a specific location using the specified services.
    Returns success status and image path or error message.
    """
    
    # Define the expected image path with location_id
    image_path = f"{image_dir}/{location_id}_google_streetview_{latitude}_{longitude}.jpg"
    
    # Download the image
    results = downloader.download_images(
        latitude=latitude,
        longitude=longitude,
        services=services,
        size=size
    )
    
    # Check if Google download was successful
    if 'google' in results and results['google'][0]:
        success, original_path = results['google']
    
        if location_id and location_id not in original_path:
            try:
                # Rename the file to include the location_id
                with open(original_path, 'rb') as src_file:
                    content = src_file.read()
                    with open(image_path, 'wb') as dest_file:
                        dest_file.write(content)
                if original_path != image_path:
                    os.remove(original_path)
                return True, image_path
            except Exception as e:
                return False, f"Error renaming file: {str(e)}"
        else:
            return True, original_path
    else:
        error_msg = results.get('google', (False, "Unknown error"))[1]
        return False, error_msg


def generate_samples_with_images(df1, zcta, downloader, points_per_zip, output_csv_path):
    """
    Generate samples with valid images for each ZIP code in df1.
    Save results to CSV in real-time.
    """
    # Check if output CSV already exists
    if os.path.exists(output_csv_path):
        all_samples = pd.read_csv(output_csv_path).to_dict('records')
        print(f"Found existing CSV with {len(all_samples)} samples. Will resume from there.")
    else:
        all_samples = []
    
    for _, row in df1.iterrows():
        zip_code = row["Zip Code"]
        
        # Check if we already have enough samples for this ZIP
        if os.path.exists(output_csv_path):
            existing_df = pd.read_csv(output_csv_path)
            zip_points = existing_df[existing_df['Zip Code'] == zip_code]
            if len(zip_points) >= points_per_zip:
                print(f"Already have {len(zip_points)} points for ZIP {zip_code}. Skipping.")
                continue
        
        poly_match = zcta[zcta["ZCTA5CE10"] == zip_code]

        if poly_match.empty:
            print(f"ZIP {zip_code} not found in shapefile. Skipping.")
            continue

        polygon = poly_match.geometry.values[0]
        
        # Generate points with images for this polygon
        valid_points = generate_points_with_images(
            polygon=polygon,
            downloader=downloader,
            zip_code=zip_code,
            city=row["City"],
            state=row["State/Region"],
            country=row["Country"],
            points_needed=points_per_zip,
            output_csv_path=output_csv_path
        )

        print(f"Collected {len(valid_points)} points with images for {row['City']} ZIP {zip_code}")
    
    # Load the final complete DataFrame
    if os.path.exists(output_csv_path):
        df_samples = pd.read_csv(output_csv_path)
        
        if 'location_id' in df_samples.columns:
            cols = df_samples.columns.tolist()
            cols.remove('location_id')
            new_cols = ['location_id'] + cols
            df_samples = df_samples[new_cols]
            df_samples.to_csv(output_csv_path, index=False)
        
        print(f"Successfully generated {len(df_samples)} total samples with images")
        return df_samples
    else:
        print("No samples were generated.")
        return pd.DataFrame()

In [ ]:
downloader = StreetViewDownloader()
samples_df1 = generate_samples_with_images(df1, zcta, downloader, points_per_zip=POINTS_PER_ZIP, output_csv_path=OUTPUT_IMAGES_CSV)

In [ ]:
# For some reasons, the resulting dataframe may include image that has not been successfully saved. 
# The following aims to remove those sampled data without valid image path.

def check_imagepath(df, base_dir=""):
    """
    Check if images exist and remove rows with missing images.
    """
    valid_rows = []
    for idx, row in df.iterrows():
        full_path = os.path.join(base_dir, row['image_path']) if base_dir else row['image_path']
        if os.path.exists(full_path):
            valid_rows.append(idx)
    
    filtered_df = df.loc[valid_rows]
    
    print(f"Total length: {len(df)}")
    print(f"Valid image path: {len(filtered_df)}")
    
    return filtered_df

samples_df1 = check_imagepath(samples_df1)
samples_df1 = samples_df1.reset_index()

### 🌍 Parameter Augmentation

This module adds geospatial and demographic context to earthquake-related datasets.

#### ⚙️ Functions

##### `add_parameter_columns()`

Adds earthquake-relevant parameters to a DataFrame using batched processing with checkpointing.

- Enriches each row with:
  - Earthquake metadata (location, magnitude)
  - Distance to epicenter
  - Closest VS30 station data
  - Census-based sociodemographics
  - Building infrastructure info

In [5]:
def add_parameter_columns(df, eq_data, max_retries=3, sleep_time=3, batch_size=10, checkpoint_file=CHECKPOINT_FILE):
    """
    Adds earthquake parameter columns to dataframe with batch processing and checkpointing.
    """
    # Load checkpoint if exists
    start_idx = 0
    result_df = df.copy().reset_index(drop=True)
    
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'rb') as f:
                checkpoint = pickle.load(f)
                result_df = checkpoint['df']
                start_idx = checkpoint['last_processed_idx'] + 1
                print(f"Resuming from index {start_idx}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
    
    # Initialize parameter columns
    parameter_columns = [
        'eq_place', 'eq_lat', 'eq_lng', 'eq_magnitude', 'eq_depth', 'distance', 'vs30',
        'population_density', 'urban_population_pct', 'median_household_income', 
        'education', 'over_65_rate', 'building'
    ]
    
    for col in parameter_columns:
        if col not in result_df.columns:
            result_df[col] = None
    
    # Pre-populate earthquake data for all rows
    result_df['eq_place'] = eq_data['place']
    result_df['eq_lat'] = round(eq_data['lat'], 3)
    result_df['eq_lng'] = round(eq_data['lng'], 3)
    result_df['eq_magnitude'] = round(eq_data['magnitude'], 1)
    result_df['eq_depth'] = round(eq_data['depth'], 1)
    
    # Process in batches
    total_rows = len(df)
    for batch_start in range(start_idx, total_rows, batch_size):
        batch_end = min(batch_start + batch_size, total_rows)
        print(f"\nProcessing batch: {batch_start} to {batch_end-1} of {total_rows}")
        
        for index in range(batch_start, batch_end):
            row = df.iloc[index]
            
            # Extract location data
            latitude = row['Latitude']
            longitude = row['Longitude']
            
            print(f"\nProcessing {index}/{total_rows}: {latitude}, {longitude}")
            
            # Get distance data
            distance = haversine_distance(latitude, longitude, eq_data['lat'], eq_data['lng'])
            result_df.loc[index, 'distance'] = round(distance, 2)

            # Get vs30 data
            result_df.loc[index, 'vs30'] = get_raster_value(src, latitude, longitude)

            # Get census data
            try:
                census = get_sociodemographics(latitude, longitude, socioeconomic_df, cbg_gdf)
                if census and census.get("status") == "success":
                    result_df.loc[index, 'population_density'] = round(census['Population_Density'], 2)
                    result_df.loc[index, 'urban_population_pct'] = round(census['Urbanized_Areas_Population_R'], 2)
                    result_df.loc[index, 'median_household_income'] = round(census['Median_income'], 0)
                    result_df.loc[index, 'education'] = round(census['Education_Degree_R'], 2)
                    result_df.loc[index, 'over_65_rate'] = round(census['Over_65_R'], 2)
                else:
                    print(f"Census data unavailable for {latitude}, {longitude}")
            except Exception as e:
                print(f"Error getting census data for {latitude}, {longitude}: {str(e)}")
                
            # Get building data
            try:
                building_result = get_building_info(latitude, longitude, radius=100, 
                                                  max_retries=max_retries, sleep_time=sleep_time)
                if building_result is not None:
                    building = describe_buildings(building_result, radius=100)
                    result_df.loc[index, 'building'] = str(building)
                else:
                    result_df.loc[index, 'building'] = "Building information is not available."
            except Exception as e:
                print(f"Error getting building info for {latitude}, {longitude}: {str(e)}")
                result_df.loc[index, 'building'] = "Building information is not available."
            
            # Save checkpoint after each row
            checkpoint = {
                'df': result_df,
                'last_processed_idx': index
            }
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint, f)
        
        # Add a pause between batches
        if batch_end < total_rows:
            pause_time = 30
            print(f"Completed batch. Pausing for {pause_time} seconds before next batch...")
            time.sleep(pause_time)
    
    return result_df

In [ ]:
result_df1 = add_parameter_columns(samples_df1, eq_data, checkpoint_file=CHECKPOINT_FILE)
result_df1

In [6]:
result_df2 = add_parameter_columns(df2, eq_data, checkpoint_file=RAG_CHECKPOINT_FILE)
result_df2


Processing batch: 0 to 9 of 553

Processing 0/553: 33.6827318987, -117.725426458

Processing 1/553: 32.7427623399, -117.126704093

Processing 2/553: 37.4720726269, -120.872713327
Attempt 1/3 failed due to insufficient response: No matching features. Check query location, tags, and log.
Retrying in 3 seconds...
Attempt 2/3 failed due to insufficient response: No matching features. Check query location, tags, and log.
Retrying in 3 seconds...
Failed to get building info after 3 attempts

Processing 3/553: 34.3502679015, -118.300760788
Attempt 1/3 failed due to insufficient response: No matching features. Check query location, tags, and log.
Retrying in 3 seconds...
Attempt 2/3 failed due to insufficient response: No matching features. Check query location, tags, and log.
Retrying in 3 seconds...
Failed to get building info after 3 attempts

Processing 4/553: 32.6680211709, -117.161121991

Processing 5/553: 34.1720242491, -116.468234152
Attempt 1/3 failed due to insufficient response: No

,City,State/Region,Country,Zip Code,MMI,Responses,Distance,Latitude,Longitude,eq_place,...,eq_magnitude,eq_depth,distance,vs30,population_density,urban_population_pct,median_household_income,education,over_65_rate,building
0,Irvine,CA,United States of America,92618,IV,48,212 km,33.682732,-117.725426,"Ridgecrest, CA",...,7.1,8.0,232.38,320,463.85,95.81,142344.0,54.95,4.41,A total of 71 buildings are found within a 100...
1,San Diego,CA,United States of America,92104,III,47,314 km,32.742762,-117.126704,"Ridgecrest, CA",...,7.1,8.0,339.4,418,10759.15,100.0,94779.0,57.8,12.39,A total of 110 buildings are found within a 10...
2,Turlock,CA,United States of America,95380,III,47,329 km,37.472073,-120.872713,"Ridgecrest, CA",...,7.1,8.0,348.07,447,140.71,38.96,36667.0,10.45,6.04,Building information is not available.
3,Sylmar,CA,United States of America,91342,IV,46,160 km,34.350268,-118.300761,"Ridgecrest, CA",...,7.1,8.0,170.3,757,0.85,7.38,68880.0,48.94,41.49,Building information is not available.
4,Coronado,CA,United States of America,92118,III,46,322 km,32.668021,-117.161122,"Ridgecrest, CA",...,7.1,8.0,347.26,94,2161.17,100.0,81821.0,39.04,0.0,A total of 13 buildings are found within a 100...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,Los Angeles,CA,United States of America,90006,IV,10,189 km,34.048026,-118.294093,"Ridgecrest, CA",...,7.1,8.0,201.69,396,19353.74,100.0,32176.0,18.31,14.88,A total of 108 buildings are found within a 10...
549,Glendale,CA,United States of America,91208,IV,10,172 km,34.196133,-118.240400,"Ridgecrest, CA",...,7.1,8.0,184.5,656,1227.68,100.0,137614.0,61.28,24.43,A total of 38 buildings are found within a 100...
550,Norco,CA,United States of America,92860,III,10,183 km,33.924663,-117.552688,"Ridgecrest, CA",...,7.1,8.0,205.24,456,3066.98,100.0,90530.0,21.48,11.9,A total of 37 buildings are found within a 100...
551,Modesto,CA,United States of America,95356,III,10,355 km,37.719492,-121.027817,"Ridgecrest, CA",...,7.1,8.0,374.56,447,141.81,42.81,68880.0,26.87,16.91,Building information is not available.


### 🌍 Prompt Generation

This module generates corresponding prompts for downstream LLM-based reasoning.


#### ⚙️ Functions

##### `generate_earthquake_prompts()`

Generates structured LLM prompts from a dataset enriched with earthquake and sociodemographic parameters.

- Initializes two prompt columns:
  - `system_prompt`: Static context prompt for LLMs
  - `earthquake_prompt`: Dynamic prompt generated from row-level variables
- Resumes prompt generation from checkpoint

In [7]:
def safe_round(value, decimals):
    if pd.isna(value) or value is None or value == "":
        return "not available"
    try:
        return round(float(value), decimals)
    except (ValueError, TypeError):
        return "not available"

# Helper function to safely format integer values
def safe_int(value):
    if pd.isna(value) or value is None or value == "":
        return "not available"
    try:
        return int(round(float(value), 0))
    except (ValueError, TypeError):
        return "not available"

# Helper function to safely get string values
def safe_str(value):
    if pd.isna(value) or value is None or value == "":
        return "not available"
    return str(value)

def generate_earthquake_prompts(df):
    """
    Generates system prompts and earthquake prompts using the parameter columns already added to the dataframe.
    """
    # Load checkpoint
    start_idx = 0
    result_df = df.copy()
    
    # Initialize prompt columns
    if 'system_prompt' not in result_df.columns:
        result_df['system_prompt'] = None
    
    if 'earthquake_prompt' not in result_df.columns:
        result_df['earthquake_prompt'] = None
    
    # Add system prompt
    if result_df['system_prompt'].isnull().all():
        result_df['system_prompt'] = SYSTEM_PROMPT
    
    total_rows = len(df)
    for index in range(start_idx, total_rows):
        row = result_df.iloc[index]
        
        # Generate earthquake prompt
        try:
            prompt_params = {
                "eq_place": safe_str(row['eq_place']),
                "eq_lat": safe_round(row['eq_lat'], 3),
                "eq_lng": safe_round(row['eq_lng'], 3),
                "eq_magnitude": safe_round(row['eq_magnitude'], 1),
                "eq_depth": safe_round(row['eq_depth'], 1),
                
                "state": safe_str(row['State/Region']),
                "city": safe_str(row['City']),
                "zipcode": safe_str(row['Zip Code']),
                "lat": safe_round(row['Latitude'], 3),
                "lng": safe_round(row['Longitude'], 3),
                "distance": safe_round(row['distance'], 2),
                "vs30": safe_str(row['vs30']),
                
                "population_density": safe_round(row['population_density'], 2),
                "urban_population_pct": safe_round(row['urban_population_pct'], 2),
                "median_household_income": safe_int(row['median_household_income']),
                "education": safe_round(row['education'], 2),
                "over_65_rate": safe_round(row['over_65_rate'], 2),
                
                "building": safe_str(row['building'])
            }
            
            result_df.loc[index, 'earthquake_prompt'] = EARTHQUAKE_PROMPT.format(**prompt_params)
            
        except Exception as e:
            print(f"Error generating prompt for index {index}: {str(e)}")
            print("Problematic row values:")
            for key, value in prompt_params.items():
                print(f"{key}: {value} (type: {type(value)})")
        
    return result_df

In [65]:
result_df1 = generate_earthquake_prompts(result_df1)
result_df1.to_csv(OUTPUT_SAMPLES_CSV,index=False)

In [8]:
result_df2 = generate_earthquake_prompts(result_df2)
result_df2.to_csv(OUTPUT_RAG_SAMPLES_CSV,index=False)